In [ ]:
import math

from qiskit_aer import AerSimulator
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

from qiskit import transpile
from qiskit.circuit import QuantumRegister, QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator

![alt text](second%20quantized.png "second quantized")
From https://doi.org/10.1063/5.0150291

Terms on the fifth line are irrelevant, they are coulomb interactions invoving the classically treated nuclei.

In [ ]:
import numpy as np
from gbasis.parsers import parse_nwchem

print("def2-SVP Basis Set Loaded from NwChem Format:")
#elec_basis_dict = parse_nwchem("6-31G(2df,p).nw")
elec_basis_dict = parse_nwchem("6-31G.nw")
nuc_basis_dict = parse_nwchem("DZSNB.nw")

"""
for atom in basis_dict:
    print(f"Atom: {atom}")
    print(f"   Number of shells: {len(basis_dict[atom])}")
    for i, shell in enumerate(basis_dict[atom]):
        print(f"   Shell {i} has angular momentum {shell[0]}")
        print(f"   Shell {i} has exponents {shell[1]}")
        print(f"   Shell {i} has coefficients {shell[2].flatten()}")
"""

In [ ]:
import numpy as np
import scipy as sp
from gbasis.parsers import parse_gbs, make_contractions
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.overlap_asymm import overlap_integral_asymmetric
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral

mass_proton = 1874.0 #mass of proton in atomic units (electron masses)

elec_atoms = ["H", "H"]
elec_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.398]])

nuc_atoms = ["Q", "Q"]
nuc_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.398]])

elec_basis = make_contractions(elec_basis_dict, elec_atoms, elec_atcoords, coord_types="cartesian")
nuc_basis = make_contractions(nuc_basis_dict, nuc_atoms, nuc_atcoords, coord_types="cartesian")

# symmetric orthogonalization of AOs. Szabo sec 3.4.5
elec_overlap = overlap_integral(elec_basis)
elec_ortho = np.linalg.inv(sp.linalg.sqrtm(elec_overlap)) # Transform needed to get orthogonal MOs

nuc_overlap = overlap_integral(nuc_basis)
nuc_ortho = np.linalg.inv(sp.linalg.sqrtm(nuc_overlap)) # Transform needed to get orthogonal MOs

elec_ke = kinetic_energy_integral(elec_basis, transform=elec_ortho)
print(elec_ke)

nuc_ke = kinetic_energy_integral(nuc_basis, transform=nuc_ortho) / mass_proton
print(nuc_ke)

elec_elec_coulomb = electron_repulsion_integral(elec_basis, transform=elec_ortho)

#test = overlap_integral(nuc_basis, transform=nuc_ortho)
#test2 = overlap_integral(elec_basis, transform=elec_ortho)
#print(test2)

In [ ]:
# Construct representation of CAR (fermionic creation/annihilation operators)
ident = [[1, 0], [0, 1]]
pauli_x = [[0, 1], [1, 0]]
pauli_y = [[0, -1j], [1j, 0]]
pauli_z = [[1, 0], [0, -1]]

modes = 2

# create representation of system of N distinguishable spins
pauli_xs = []
pauli_ys = []
pauli_zs = []

spin_raise = []
spin_lower = []

create = []
annihilate = []

for i in range(modes):
    x = y = z = [1]
    for j in range(modes):
        if i == j:
            x = np.kron(x, pauli_x)
            y = np.kron(y, pauli_y)
            z = np.kron(z, pauli_z)
        else:
            x = np.kron(x, ident)
            y = np.kron(y, ident)
            z = np.kron(z, ident)

    pauli_xs.append(x)
    pauli_ys.append(y)
    pauli_zs.append(z)

    spin_raise.append(0.5*(x+1j*y))
    spin_lower.append(0.5*(x-1j*y))

# construct fermionic creation/annihilation operators via JWT
for i in range(modes):
    sum = np.zeros([2**modes, 2**modes])
    for j in range(i):
        sum = sum + (spin_raise[j] @ spin_lower[j])

    create_op = sp.linalg.expm(1j*math.pi*sum) @ spin_raise[i]
    annihilate_op = sp.linalg.expm(-1j*math.pi*sum) @ spin_lower[i]

    annihilate.append(annihilate_op)
    create.append(create_op)

for i in range(modes):
    for j in range(modes):
        #commutator = annihilate[i] @ annihilate[j] + annihilate[j] @ annihilate[i]
        commutator = create[i] @ create[j] + create[j] @ create[i]
        #commutator = annihilate[i] @ create[j] + create[j] @ annihilate[i]
        #print(commutator)

        if(np.allclose(commutator, np.eye(2**modes))):
            print(1)
        elif(np.allclose(commutator, np.zeros([2**modes, 2**modes]))):
            print(0)
        else:
            print("?")

print("bruh")


In [ ]:
import math
import numpy as np
from gbasis.parsers import parse_nwchem

elec_basis_dict = parse_nwchem("6-31G.nw")
nuc_basis_dict = parse_nwchem("DZSNB.nw")

"""
for atom in basis_dict:
    print(f"Atom: {atom}")
    print(f"   Number of shells: {len(basis_dict[atom])}")
    for i, shell in enumerate(basis_dict[atom]):
        print(f"   Shell {i} has angular momentum {shell[0]}")
        print(f"   Shell {i} has exponents {shell[1]}")
        print(f"   Shell {i} has coefficients {shell[2].flatten()}")
"""
import numpy as np
import scipy as sp
from gbasis.parsers import parse_gbs, make_contractions
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.overlap_asymm import overlap_integral_asymmetric
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral

mass_proton = 1874.0  #mass of proton in atomic units (electron masses)

# Centers of electron orbitals (just the H atoms position)
elec_atoms = ["H", "H"]
elec_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.398]])

# Centers of nuclear orbitals for protons
nuc_atoms = ["Q", "Q"]
nuc_atcoords = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 1.398]])


# Construct full molecular orbital  basis from atomic orbitals centered at the positions "atcoords"
elec_basis = make_contractions(elec_basis_dict, elec_atoms, elec_atcoords, coord_types="cartesian")
nuc_basis = make_contractions(nuc_basis_dict, nuc_atoms, nuc_atcoords, coord_types="cartesian")
full_basis = elec_basis + nuc_basis # Bases are tuples of contracted gaussian objects. NOTE that the overlap

In [ ]:
# symmetric orthogonalization of AOs. Szabo sec 3.4.5
elec_overlap = overlap_integral(elec_basis)
elec_ortho = np.linalg.inv(sp.linalg.sqrtm(elec_overlap))  # Transform needed to get orthogonal MOs

nuc_overlap = overlap_integral(nuc_basis)
nuc_ortho = np.linalg.inv(sp.linalg.sqrtm(nuc_overlap))  # Transform needed to get orthogonal MOs

full_ortho = np.block([[elec_ortho, np.zeros((len(elec_basis), len(nuc_basis)))],
                       [np.zeros((len(nuc_basis), len(elec_basis))), nuc_ortho]])

full_overlap = overlap_integral(full_basis, transform=full_ortho)
print(np.matrix.round(full_overlap, 2))
# Note the "non-zero" overlap between nuclear and electronic orbitals. They actually are orthogonal in the Hilbert space though.


In [ ]:
c = 1/math.sqrt(2)
h = [[c, c],
     [c, -c]]
symmetrize = np.kron(h,np.eye(2))
print(symmetrize)

sym_overlap = overlap_integral(elec_basis, transform=symmetrize) # Transformation to symmetric/antisymmetric orbitals. No longer unit length
sym_ortho = np.linalg.inv(sp.linalg.sqrtm(sym_overlap)) @ symmetrize

print(np.matrix.round(overlap_integral(elec_basis, transform=sym_ortho), 21))
print(np.matrix.round(kinetic_energy_integral(elec_basis, transform=sym_ortho), 2))



#print(np.matrix.round(sym_overlap, 2))
#print(np.matrix.round(sym_ortho, 2))

In [ ]:
elec_ke = kinetic_energy_integral(elec_basis, transform=elec_ortho)
print(elec_ke)

nuc_ke = kinetic_energy_integral(nuc_basis, transform=nuc_ortho) / mass_proton
print(nuc_ke)

coulomb = electron_repulsion_integral(full_basis, transform=full_ortho)



In [ ]:
print(coulomb[1,3,1,22]) # uses physicist notation:

# Physicist notation integral
![](physnot.png)
this is coulomb[a,b,c,d]

In [ ]:
elec_modes = 2*len(elec_basis)
nuc_modes = 2*len(nuc_basis)

# KE of electron and nuclei in terms of annihilation/creation ops
elec_KE_fermion_op = FermionicOp({}, num_spin_orbitals=elec_modes)
nuc_KE_fermion_op = FermionicOp({}, num_spin_orbitals=nuc_modes)

# SparsePauliOp identity for  qubits representing the electronic modes
elec_pauli_identity = SparsePauliOp("I"*elec_modes)
nuc_pauli_identity = SparsePauliOp("I"*nuc_modes)

mapper = JordanWignerMapper()



for i in range(len(elec_basis)):
    for j in range(len(elec_basis)):
        # Only include terms with the same spin: 2i, 2j and 2i+1, 2j+1 (alpha and beta orbitals respectively)
        elec_KE_fermion_op += FermionicOp({f"+_{2*i} -_{2*j}": elec_ke[i, j]}, num_spin_orbitals=elec_modes)
        elec_KE_fermion_op += FermionicOp({f"+_{2*i+ 1} -_{2*j + 1}": elec_ke[i, j]}, num_spin_orbitals=elec_modes)

for i in range(len(nuc_basis)):
    for j in range(len(nuc_basis)):
        # Only include terms with the same spin: 2i, 2j and 2i+1, 2j+1 (alpha and beta orbitals respectively)
        nuc_KE_fermion_op += FermionicOp({f"+_{2*i} -_{2*j}": nuc_ke[i, j]}, num_spin_orbitals=nuc_modes)
        nuc_KE_fermion_op += FermionicOp({f"+_{2*i + 1} -_{2*j + 1}": nuc_ke[i, j]}, num_spin_orbitals=nuc_modes)


#print(elec_KE_fermion_op)
#print(nuc_KE_fermion_op)

elec_KE_pauli_op = mapper.map(elec_KE_fermion_op) ^ nuc_pauli_identity
nuc_KE_pauli_op = elec_pauli_identity ^ mapper.map(nuc_KE_fermion_op)

KE_pauli_op = elec_KE_pauli_op + nuc_KE_pauli_op

print(KE_pauli_op)


In [ ]:
elec_elec_coulomb_fermion_op = FermionicOp({}, num_spin_orbitals=elec_modes)
nuc_nuc_coulomb_fermion_op = FermionicOp({}, num_spin_orbitals=nuc_modes)

for i in range(len(elec_basis)):
    for j in range(len(elec_basis)):
        for k in range(len(elec_basis)):
            for l in range(len(elec_basis)):
                for spin1 in range(2):
                    for spin2 in range(2):
                        elec_elec_coulomb_fermion_op += FermionicOp({
                            f"+_{2*i + spin1} +_{2*j + spin2} -_{2*k + spin1} -_{2*l + spin2}": 0.5 * coulomb[i, j, l, k],
                        }, num_spin_orbitals=elec_modes)

for i in range(len(nuc_basis)):
    for j in range(len(nuc_basis)):
        for k in range(len(nuc_basis)):
            for l in range(len(nuc_basis)):
                for spin1 in range(2):
                    for spin2 in range(2):
                        nuc_nuc_coulomb_fermion_op += FermionicOp({
                            f"+_{2*i + spin1} +_{2*j + spin2} -_{2*k + spin1} -_{2*l + spin2}": 0.5 * coulomb[i + len(elec_basis), j + len(elec_basis), l + len(elec_basis), k + len(elec_basis)],
                        }, num_spin_orbitals=elec_modes)

# cant combine elec and nuc fermion ops or else the jordan wigner mapper will think theyre anticommuting (they commute. indistinguishable!)
# so i will have to convert the 2 one body ops (one nuclear and one electronic) to qubit via JWT, SEPARATELY, then combine them (tensor product)

elec_elec_coulomb_pauli_op = mapper.map(elec_elec_coulomb_fermion_op)
nuc_nuc_coulomb_pauli_op = mapper.map(nuc_nuc_coulomb_fermion_op)


elec_nuc_coulomb_pauli_op = 0

for i in range(len(nuc_basis)):
    for j in range(len(elec_basis)):
        for k in range(len(nuc_basis)):
            for l in range(len(elec_basis)):
                for spin1 in range(2):
                    for spin2 in range(2):
                        elec_fermion_op = FermionicOp({f"+_{2*i + spin1} -_{2*k + spin1}": 1,}, num_spin_orbitals=elec_modes)
                        nuc_fermion_op = FermionicOp({f"+_{2*j + spin2} -_{2*l + spin2}": 1,}, num_spin_orbitals=nuc_modes)

                        elec_nuc_coulomb_pauli_op += coulomb[i, j + len(elec_basis), k, l + len(elec_basis)] * (mapper.map(elec_fermion_op) ^ mapper.map(nuc_fermion_op))
                        break
                    break
                break
            break
        break
    break


print(elec_nuc_coulomb_pauli_op)


In [ ]:
print(coulomb[0, 4, 0, 4])

In [ ]:

#test = overlap_integral(nuc_basis, transform=nuc_ortho)
#test2 = overlap_integral(elec_basis, transform=elec_ortho)
#print(test2)
# Construct representation of CAR (fermionic creation/annihilation operators)
ident = [[1, 0], [0, 1]]
pauli_x = [[0, 1], [1, 0]]
pauli_y = [[0, -1j], [1j, 0]]
pauli_z = [[1, 0], [0, -1]]

modes = 2

# create representation of system of N distinguishable spins
pauli_xs = []
pauli_ys = []
pauli_zs = []

spin_raise = []
spin_lower = []

create = []
annihilate = []

for i in range(modes):
    x = y = z = [1]
    for j in range(modes):
        if i == j:
            x = np.kron(x, pauli_x)
            y = np.kron(y, pauli_y)
            z = np.kron(z, pauli_z)
        else:
            x = np.kron(x, ident)
            y = np.kron(y, ident)
            z = np.kron(z, ident)

    pauli_xs.append(x)
    pauli_ys.append(y)
    pauli_zs.append(z)

    spin_raise.append(0.5 * (x + 1j * y))
    spin_lower.append(0.5 * (x - 1j * y))

# construct fermionic creation/annihilation operators via JWT
for i in range(modes):
    sum = np.zeros([2 ** modes, 2 ** modes])
    for j in range(i):
        sum = sum + (spin_raise[j] @ spin_lower[j])

    create_op = sp.linalg.expm(1j * math.pi * sum) @ spin_raise[i]
    annihilate_op = sp.linalg.expm(-1j * math.pi * sum) @ spin_lower[i]

    annihilate.append(annihilate_op)
    create.append(create_op)

for i in range(modes):
    for j in range(modes):
        #commutator = annihilate[i] @ annihilate[j] + annihilate[j] @ annihilate[i]
        commutator = create[i] @ create[j] + create[j] @ create[i]
        #commutator = annihilate[i] @ create[j] + create[j] @ annihilate[i]
        #print(commutator)

        if (np.allclose(commutator, np.eye(2 ** modes))):
            print(1)
        elif (np.allclose(commutator, np.zeros([2 ** modes, 2 ** modes]))):
            print(0)
        else:
            print("?")

print("bruh")



In [ ]:
print(type(elec_basis))z